In [2]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report, confusion_matrix, precision_score

In [3]:
import sklearn
sklearn.set_config(display='text')

In [4]:
# 1. 데이터 로드
df = joblib.load('../quality_data/quality_prep.pkl')

In [5]:
# 2. 저장된 변수들 추출
X_train = df['X_train']
X_test = df['X_test']
y_train = df['y_train']
y_test = df['y_test']
preprocessor = df['preprocessor']  

In [6]:
# 3. 랜덤 포레스트 분류기 생성 및 학습
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',   
    n_jobs=-1
)

model.fit(X_train, y_train)

print('===학습 완료===')

===학습 완료===


In [7]:
# 4. 예측 및 평가
y_pred = model.predict(X_test)

# 다중 클래스(High, Medium, Low) 평가이므로 average='weighted' 지정
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print("=== 모델 정확도 ===")
print(f" Accuracy  (정확도) : {acc:.4f}")

print("=== 핵심 성능 지표 ===")
print(f"1. Accuracy  (정확도) : {acc:.4f}")
print(f"2. Precision (정밀도) : {prec:.4f}")
print(f"3. Recall    (재현율) : {rec:.4f}")
print(f"4. F1-Score  (F1점수) : {f1:.4f}\n")

=== 모델 정확도 ===
 Accuracy  (정확도) : 1.0000
=== 핵심 성능 지표 ===
1. Accuracy  (정확도) : 1.0000
2. Precision (정밀도) : 1.0000
3. Recall    (재현율) : 1.0000
4. F1-Score  (F1점수) : 1.0000



In [8]:
# 평가 출력
print("=== 분류 결과 상세 리포트 ===")
print(classification_report(y_test, y_pred))

=== 분류 결과 상세 리포트 ===
              precision    recall  f1-score   support

        High       1.00      1.00      1.00        36
         Low       1.00      1.00      1.00         2
      Medium       1.00      1.00      1.00        37

    accuracy                           1.00        75
   macro avg       1.00      1.00      1.00        75
weighted avg       1.00      1.00      1.00        75



In [9]:
# Confusion Matrix 텍스트
# label 순서 추출
labels = model.classes_
cm = confusion_matrix(y_test, y_pred, labels=labels)

print("=== 혼동 행렬 (Confusion Matrix) ===")
cm_df = pd.DataFrame(cm, index=[f"Actual_{l}" for l in labels], columns=[f"Pred_{l}" for l in labels])
print(cm_df)
print("\n")

=== 혼동 행렬 (Confusion Matrix) ===
               Pred_High  Pred_Low  Pred_Medium
Actual_High           36         0            0
Actual_Low             0         2            0
Actual_Medium          0         0           37




In [10]:
import os
#  파일 저장 경로 설정 
current_dir = os.path.dirname(os.path.abspath('quality_model_train.ipynb')) if '__file__' not in globals() else os.path.dirname(os.path.abspath(__file__))
csv_path = os.path.join(current_dir, 'quality_confusionmatrix.csv')

#  Confusion Matrix DataFrame 생성 및 CSV 저장
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{l}" for l in labels],
    columns=[f"Pred_{l}" for l in labels]
)
cm_df.to_csv(csv_path, index=True, encoding='utf-8-sig')

print(f"=== CSV 저장 완료: {csv_path} ===")

=== CSV 저장 완료: c:\sleep\data\quality_testcode\quality_confusionmatrix.csv ===


In [11]:
#  파일 저장 경로 설정 
current_dir = os.path.dirname(os.path.abspath('quality_model_train.ipynb')) if '__file__' not in globals() else os.path.dirname(os.path.abspath(__file__))
py_path = os.path.join(current_dir, 'quality_model_train.py')

#  Jupyter Notebook 명령어로 .py 파일 변환 실행
!jupyter nbconvert --to script quality_model_train.ipynb --output-dir="{current_dir}"

print(f"=== .py 파일 생성 완료 ===")
print(f"생성 경로: {py_path}")

=== .py 파일 생성 완료 ===
생성 경로: c:\sleep\data\quality_testcode\quality_model_train.py


usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: kernel kernelspec migrate run troubleshoot

Jupyter command `jupyter-nbconvert` not found.


In [12]:
# 파일 저장 경로 설정 
current_dir = os.path.dirname(os.path.abspath('quality_model_train.ipynb')) if '__file__' not in globals() else os.path.dirname(os.path.abspath(__file__))
model_dir = os.path.normpath(os.path.join(current_dir, '..', '..', 'model'))

# model 폴더가 없으면 자동 생성
os.makedirs(model_dir, exist_ok=True)

# PKL 저장 경로 지정
model_pkl_path = os.path.join(model_dir, 'quality_model_rf.pkl')

#  PKL 파일 저장
final_info = {
    'model': model,
    'preprocessor': preprocessor if 'preprocessor' in globals() else None,
    'labels': labels
}
joblib.dump(final_info, model_pkl_path)

print(f"=== PKL 저장 완료 ===")
print(f"저장 위치: {model_pkl_path}")

=== PKL 저장 완료 ===
저장 위치: c:\sleep\model\quality_model_rf.pkl


In [13]:
print("y_train 클래스별 개수:\n", pd.Series(y_train).value_counts())
print("y_test 클래스별 개수:\n", pd.Series(y_test).value_counts())

y_train 클래스별 개수:
 Quality of Sleep
Medium    145
High      144
Low        10
Name: count, dtype: int64
y_test 클래스별 개수:
 Quality of Sleep
Medium    37
High      36
Low        2
Name: count, dtype: int64
